# 164 — Head Value Analysis

Analyze what attention heads write to the residual stream: value vector
structure, output decomposition, attention-weighted contributions,
OV effective rank, and writing directions.

## Why This Matters

Attention head value reveals how heads route information between positions. Understanding attention mechanics is central to reverse-engineering transformer circuits, as attention patterns determine what information each component can access and transform.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.head_value_analysis import (
    value_vector_analysis,
    head_output_decomposition,
    value_weighted_attention,
    value_rank_analysis,
    head_writing_direction,
)

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.arange(15)

In [ ]:
result = value_vector_analysis(model, tokens, layer=0)
for h in result['per_head']:
    print(f"Head {h['head']}: norm={h['value_norm']:.4f}  "
          f"pos_var={h['position_variation']:.4f}  "
          f"centroid_cos={h['mean_cosine_to_centroid']:.3f}")

In [ ]:
result = head_output_decomposition(model, tokens, layer=0)
print(f"Top predicted token: {result['top_token']}\n")
for h in result['per_head']:
    print(f"Head {h['head']}: out_norm={h['output_norm']:.4f}  "
          f"logit_contrib={h['logit_contribution']:+.4f}  "
          f"unembed_align={h['unembed_alignment']:.3f}")

In [ ]:
result = value_weighted_attention(model, tokens, layer=0)
for h in result['per_head']:
    top3 = h['source_contributions'][:3]
    sources = ', '.join(f"pos{s['position']}({s['attention_weight']:.2f})" for s in top3)
    print(f"Head {h['head']}: dominant_src={h['dominant_source']}  "
          f"conc={h['concentration']:.3f}  top3=[{sources}]")

In [ ]:
result = value_rank_analysis(model, layer=0)
for h in result['per_head']:
    print(f"Head {h['head']}: eff_rank={h['effective_rank']:.1f}  "
          f"top_sv={h['top_singular_value']:.4f}  "
          f"cond={h['condition_number']:.0f}")

In [ ]:
result = head_writing_direction(model, tokens, layer=0, top_k=5)
for h in result['per_head']:
    tokens_str = ', '.join(f"tok{a['token']}({a['logit_contribution']:+.3f})" 
                          for a in h['top_token_alignments'])
    print(f"Head {h['head']}: norm={h['output_norm']:.4f}  promotes=[{tokens_str}]")